In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [6]:
from pathlib import Path
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


# Data loading + transforms (PyTorch)

In [7]:
DATA_ROOT = Path("/kaggle/input/chest-xray-pneumonia/chest_xray")  # adapter si besoin

In [9]:
# 1. Transforms
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [10]:
# 2. Datasets
train_dir = DATA_ROOT / "train"
val_dir = DATA_ROOT / "val"
test_dir = DATA_ROOT / "test"

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(val_dir,   transform=test_transform)
test_dataset  = datasets.ImageFolder(test_dir,  transform=test_transform)

class_names = train_dataset.classes
print("Classes:", class_names)  # ['NORMAL', 'PNEUMONIA']

Classes: ['NORMAL', 'PNEUMONIA']


In [14]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

5216
16
624


In [15]:
# 3. DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Modèle CNN avancé avec Transfer Learning

In [16]:
def build_model(num_classes: int = 2, fine_tune: bool = False):
    # ResNet18 pré-entraîné
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # Si on ne fait pas de fine-tuning complet : on freeze les features
    if not fine_tune:
        for param in model.parameters():
            param.requires_grad = False

    # On remplace la fully connected finale
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )

    return model.to(device)

model = build_model(num_classes=len(class_names), fine_tune=False)
print(model.fc)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 163MB/s] 


Sequential(
  (0): Linear(in_features=512, out_features=256, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=256, out_features=2, bias=True)
)


# Gérer le déséquilibre des classes
Il y a beaucoup plus de PNEUMONIA que de NORMAL. Pour être un peu sérieux :

In [17]:
# Compter les classes dans le train set
labels = [y for _, y in train_dataset]
labels = np.array(labels)

class_counts = np.bincount(labels)
print("Samples par classe:", class_counts)

# Poids inverses
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

Samples par classe: [1341 3875]


# Optimizer, scheduler, training loop

In [18]:
# Optimizer et scheduler
learning_rate = 1e-3
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                 factor=0.5, patience=2)

NUM_EPOCHS = 10
best_val_loss = float("inf")
best_model_path = Path("/kaggle/working/best_model.pt")
best_model_path.parent.mkdir(parents=True, exist_ok=True)


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        _, preds = torch.max(outputs, 1)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data).item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = running_corrects / len(loader.dataset)
    return epoch_loss, epoch_acc


def eval_one_epoch(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    running_corrects = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data).item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = running_corrects / len(loader.dataset)
    return epoch_loss, epoch_acc


for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = eval_one_epoch(model, val_loader, criterion)

    scheduler.step(val_loss)

    print(f"[{epoch+1}/{NUM_EPOCHS}] "
          f"Train loss: {train_loss:.4f} | acc: {train_acc:.4f} | "
          f"Val loss: {val_loss:.4f} | acc: {val_acc:.4f}")

    # Early best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print("→ Nouveau meilleur modèle sauvegardé.")

[1/10] Train loss: 0.2662 | acc: 0.8913 | Val loss: 0.3207 | acc: 0.8750
→ Nouveau meilleur modèle sauvegardé.
[2/10] Train loss: 0.1887 | acc: 0.9283 | Val loss: 0.2491 | acc: 0.9375
→ Nouveau meilleur modèle sauvegardé.
[3/10] Train loss: 0.2071 | acc: 0.9195 | Val loss: 0.1702 | acc: 1.0000
→ Nouveau meilleur modèle sauvegardé.
[4/10] Train loss: 0.1804 | acc: 0.9314 | Val loss: 0.1725 | acc: 0.9375
[5/10] Train loss: 0.1509 | acc: 0.9396 | Val loss: 0.5729 | acc: 0.7500
[6/10] Train loss: 0.1606 | acc: 0.9383 | Val loss: 0.3148 | acc: 0.9375
[7/10] Train loss: 0.1441 | acc: 0.9419 | Val loss: 0.3562 | acc: 0.8750
[8/10] Train loss: 0.1409 | acc: 0.9454 | Val loss: 0.2830 | acc: 0.8750
[9/10] Train loss: 0.1455 | acc: 0.9429 | Val loss: 0.4590 | acc: 0.7500
[10/10] Train loss: 0.1434 | acc: 0.9446 | Val loss: 0.2479 | acc: 0.9375


# Évaluation finale sur le test set

In [19]:
# Charger le meilleur modèle
best_model = build_model(num_classes=len(class_names), fine_tune=False)
best_model.load_state_dict(torch.load(best_model_path, map_location=device))

best_model.eval()
all_labels = []
all_preds = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = best_model(inputs)
        _, preds = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

print("Classification report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

print("Confusion matrix:")
print(confusion_matrix(all_labels, all_preds))


Classification report:
              precision    recall  f1-score   support

      NORMAL       0.86      0.85      0.86       234
   PNEUMONIA       0.91      0.92      0.91       390

    accuracy                           0.89       624
   macro avg       0.89      0.89      0.89       624
weighted avg       0.89      0.89      0.89       624

Confusion matrix:
[[200  34]
 [ 33 357]]
